Antonio Esteves @ UMinho, Jan 2024

# 12.1 A fairground game

At a fairground a man advertises a gambling game that allows participants the chance to win a monetary prize, if they pay an entrance fee. The game sequence goes like this:
* You pay `X` £.
* The man flips a fair coin, with an equal chance of the coin landing heads or tails up,
* If the coin lands tails up, the game ends and you walk away with nothing.
* If the coin lands heads up, he flips the coin a further two times and you receive the total number of heads across these latter two flips, `H`. So if the coin lands heads up twice, you receive 2 £; if once, you receive 1 £; if zero, you receive 0 £.
* Your winnings are given by (H - X) £.

**Problem 12.1.1**. Calculate the expected value of your winnings `W` if you participate, and hence determine the fair price of the game.


In [ ]:
# The expected value of winnings W is given by:

E_W = 0.5 * 0 + 0.5 * (0.25*0 + 0.5*1 +0.25*2) - X = 0.5 * (0.5+0.5) - X = 0.5 - X

# So, the fair price is: 

0.5 - X = 0 <=> X = 0.5 £


**Problem 12.1.2**. Create a function that simulates a single run of the game, and use this to estimate the expected value of your winnings. Hint: use scipy `binom` distribution.

In [ ]:
from scipy.stats import binom
from statistics  import mean

def run_fairground_game(p_coin=0.5,fee_X=0.5):
    h1 = binom.rvs(n=1,p=p_coin,size=1)
    if h1[0]==0:
        W = h1[0] - fee_X
    else:
        h2 = binom.rvs(n=2,p=p_coin,size=1)
        W = h2[0] - fee_X
    return W


In [ ]:
# run once

w = run_fairground_game(0.5,0.5)
print(f'Win: {w}')
      
# rum 1000x to compute the mean

w_list = []
for n in range(1000):
    w = run_fairground_game(0.5,0.5)
    w_list.append(w)
#print(w_list)
print(f'mean win: {mean(w_list)}')

**Problem 12.1.3**. Suppose that you pay 1 £ for each game, and start with 10 £ in your pocket. By using your previously-created function, or otherwise, determine the expected number of games you can play before going broke.

In [ ]:
w_list = []
for n in range(1000):
    w = run_fairground_game(0.5,1.0)
    w_list.append(w)

mean_w = mean(w_list)
print(f'mean win: {mean_w}')

In [ ]:
initial_amount   = 10
if mean_w < 0:
    E_N_zero_balance = initial_amount/round(abs(mean_w),1)
    print(f'Expected number of runs before balance is zero: {E_N_zero_balance:.0f}')
else:
    print('mean win is positive, so our balance will not be zero')

**Problem 12.1.4**. Suppose you start with 10 £, and play the game 200 times, stopping only if your balance is below the price of entry, each time paying 0.55 £. You want to insure against the risk of losing all your wealth. What is the fair price to pay for such an insurance scheme?

In [ ]:
'''
The fair price to pay is the probability that we go broke x the loss that it causes (10 £).
To determine this risk we can use sampling, creating a function that determines 
the number of times we can play before we go broke.
'''

def play_200_games_or_broke(p, fee, amount) -> int:
    n = 0
    while amount>fee:
        w = run_fairground_game(p,fee)
        amount += w
        n += 1
    return n

n_games_played = play_200_games_or_broke(0.5, 0.55, 10)
print(f'number of played games: {n_games_played}')

In [ ]:
# rum 1000x to compute the mean

amount  = 10
N_RUNS  = 1000
n_broke = 0
for n in range(N_RUNS):
    n = play_200_games_or_broke(0.5, 0.55, amount)
    if n<100:
        n_broke +=1

percent_broke = (n_broke)/N_RUNS
print(f'percentage of times we go broke in 200 games: {percent_broke*100}%')
print(f'we should pay for insurance {percent_broke*amount} ')

# 12.2  Independent sampling

An analysis results in a posterior with the following probability density function:

$f(x) = \frac{1}{1.33485} \frac{e^{-x^2/1}}{\sqrt{2 \pi}},\ \ if\ x<0.9735$ <P>
      $\ \ \ \ \ \ \ \ =  0.186056,\ \ if\ 0.9735 \leq x \leq 5$ <P>
      $\ \ \ \ \ \ \ \ = 0, \ \ otherwise$ <P>

**Problem 12.2.1**. Verify that this is a valid PDF over the range [0:8]. (hint: use integration)

In [ ]:
from scipy.integrate import quad
import numpy as np

def pdf(x):
    if x<0.9735:
        exp2  = np.exp((-x*x)/2.0)
        frac2 = exp2 / np.sqrt(2*np.pi)
        fx    = 1.0/1.33485
        fx   *= frac2
    elif x>=0.9375 and x<=5:
        fx = 0.186056
    else:
        fx = 0
    return fx


In [ ]:
# quad() arguments: function, inferior limit, superior limit, arguments of function
# quad() returns: tuple (integral value, integral error)
I_pdf, I_pdf_error = quad(pdf, 0, 8)
print(f'Integral of the PDF: {I_pdf:.3f} Error: {I_pdf_error}') 

**Problem 12.2.2**. Using independent sampling estimate the mean and variance of this distribution.

There are a large number of methods. The first is **rejection sampling** where we generate two (pseudo)random sets of points from the uniform distribution: $x \in (0, 8)$ and 
$y \in (0, 1/(1.335*2*\pi))$. We accept the point as a sample from our distribution if: $y \leq pdf(x)$.

In [ ]:
def reject_sampling (func, NUM):
    y_max = 1.0 / (1.335 * np.sqrt(2.0 * np.pi))
    x_arr = np.random.default_rng().uniform(low=0, high=8, size=NUM)
    y_arr = np.random.default_rng().uniform(low=0, high=y_max, size=NUM)
    samples_less = []
    for n in range(NUM):
        if(y_arr[n] < func(x_arr[n])):
            samples_less.append(x_arr[n])
    return(samples_less)

sample = reject_sampling (pdf, 10000)
sample_mean = np.mean(sample)
sample_var  = np.var(sample)
print(f'function mean:     {sample_mean:.4f}')
print(f'function variance: {sample_var:.4f}')

**Problem 12.2.4**. Verify your previous answer by calculating the mean and variance of this distribution. This can be done using numerical integration.

In [ ]:
# mean      = INTEGRAL x.f(x).dx
# variance  = INTEGRAL (x-mean)^2.f(x).dx

def f_mean(x):
    x_fx = x * pdf(x)
    return x_fx

def f_var(x, x_mean):
    x_var_exp = (x-x_mean)*(x-x_mean)*pdf(x)
    return x_var_exp

# quad() arguments: function, inferior limit, superior limit, arguments of function
# quad() returns: tuple (integral value, integral error)

mean_I, mean_I_error = quad(f_mean, 0, 8)
print(f'Mean computed with integral of x*pdf(x): {mean_I:.3f} Error: {mean_I_error}') 

var_I, var_I_error = quad(f_var, 0, 8, args=mean_I)
print(f'Variance computed with integral: {var_I:.3f} Error: {var_I_error}') 

# 12.4 Markovian coin

Consider a type of coin for which the result of the next throw (heads or tails) can depend on the result of the current throw. In particular if a "heads" is thrown then the probability of obtaining a "heads" on the next throw is $(\frac{1}{2} + \epsilon)$; if instead a tails is thrown then the probability of obtaining a "tails" on the next throw is $(\frac{1}{2} + \epsilon)$. To start, we assume $0 \leq \epsilon \leq \frac{1}{2}$. The random variable $X \in \{0,1\}$ if the coin lands "tails-up" or "heads-up" on a given throw.

**Problem 12.4.1**. Find the mean of the coin supposing it starts with probability $\frac{1}{2}$ on each side.

$E(X1 | X0 ) = Pr(X1=1 | X0) * 1 + Pr(X1=0 | X0) * 0$ <P>
            $= Pr(X1=1 | X0)$ <P>

By the law of iterated expectations: <P>
    
$E(X1) = E[E(X1 | X0)]$ <P>
$      = Pr(X0=0) * Pr(X1=1 | X0=0) + Pr(X0=1) * Pr(X1=1 | X0=1)$ <P>
$      = 1/2 * (1/2 − \epsilon) + 1/2 * (1/2 + \epsilon)$ <P>
$      = 1/2$ <P>

Therefore we know that $E(X2) = 1/2$, and ... $E(Xk) = 1/2$. <P>

# 13.2 The fairground revisited

You again find yourself in a fairground, and where there is a stall offering the chance to win money if you participate in a game. Before participating you watch a few other plays of the game (by other people in the crowd) to try to determine whether you want to play.

**Problem 13.2.1**. In the most-boring version of the game, a woman flips a coin and you bet on its outcome. If the coin lands heads-up, you win; if tails, you lose. Based on your knowledge of similar games (and knowledge that the game must be rigged for the woman to make a profit!) you assume that the coin must be biased towards tails. As such you decide to specify a prior on the probability of the coin falling heads-up as θ ∼ beta(2, 5). Graph this function, and – using your knowledge of the beta distribution – determine the mean parameter value specified by this prior.

In [ ]:
import numpy as np
from   scipy.stats import beta
import matplotlib.pyplot as plt

In [ ]:
def prior_beta(theta, a, b):
    return beta.pdf(theta, a, b)

In [ ]:
a, b = 2.0, 5.0

mean, var, skew, kurt = beta.stats(a, b, moments='mvsk')

print(f'Mean beta(2,5) = {mean}')
print(f'2/7            = {2/7}')

In [ ]:
fig, ax = plt.subplots(1, 1)
x = np.linspace(beta.ppf(0.01, a, b), beta.ppf(0.99, a, b), 100)
ax.plot(x, beta.pdf(x, a, b), 'r-', lw=5, alpha=0.6, label='beta pdf')

**Problem 13.2.2**. You watch the last 10 plays of the game, and the outcome is heads 3/10 times.
Assuming a binomial likelihood, create a function that determines the likelihood for a given value of the probability of heads, θ. Hence or otherwise, determine the maximum likelihood estimate of θ.

In [ ]:
from scipy.stats import binom
from statistics  import mean

def likelihood_binom(k, n, theta):
    return binom.pmf(k, n, theta)

In [ ]:
def plot_binom(n, theta):
    fig, ax = plt.subplots(1, 1)
    k = np.arange(binom.ppf(0.01, n, theta), binom.ppf(0.99, n, theta))
    print(k)
    ax.plot(k, binom.pmf(k, n, theta), 'bo', ms=8, label='binom pmf')
    ax.vlines(k, 0, binom.pmf(k, n, theta), colors='b', lw=5, alpha=0.5)
    ax.set_ylabel(f'binom(k,{n},{theta})', fontsize=11)
    ax.set_xlabel('k', fontsize=11)

In [ ]:
plot_binom(10, 0.3)

In [ ]:
def plot_likelihood(k=3, n=10):
    fig, ax = plt.subplots(1, 1)
    theta = np.linspace(0, 1, 100)
    ax.plot(theta, binom.pmf(k, n, theta), 'bo', ms=8, label='binom pmf')
    ax.set_ylabel('likelihood = p(data|theta)', fontsize=11)
    ax.set_xlabel('theta', fontsize=11)

In [ ]:
plot_likelihood(3, 10)

In [ ]:
def mle_likelihood(k=3, n=10):
    NUM=1000
    theta   = np.linspace(0, 1, NUM)
    p_theta = binom.pmf(k, n, theta)
    max     = p_theta[0]
    id_max  = 0
    for i in range(NUM):
        if p_theta[i]> max:
            max = p_theta[i]
            id_max = i
    mle_theta = id_max/NUM
    return mle_theta

In [ ]:
mle_theta = mle_likelihood(3, 10)
print(f'MLE theta: {mle_theta}')

**Problem 13.2.3**. Graph the likelihood × prior. From the graph approximately determine the
MAP (=mode) θ estimate value.

In [ ]:
def prior_x_likelihood(theta, a, b, k, n):
    prior_likelihood = prior_beta(theta, a, b) * likelihood_binom(k, n, theta)
    return prior_likelihood

In [ ]:
def plot_prior_x_likelihood(a=2, b=5, k=3, n=10):
    fig, ax = plt.subplots(1, 1)
    theta = np.linspace(0, 1, 100)
    ax.plot(theta, prior_x_likelihood(theta, a, b, k, n), 'bo', ms=8, label='prior x likelihood')
    ax.set_ylabel('p(theta) x p(data|theta)', fontsize=11)
    ax.set_xlabel('theta', fontsize=11)

In [ ]:
plot_prior_x_likelihood(2, 5, 3, 10)

In [ ]:
def map_prior_x_likelihood(a=2, b=5, k=3, n=10):
    NUM=1000
    theta   = np.linspace(0, 1, NUM)
    p_theta = prior_x_likelihood(theta, a, b, k, n)
    max     = p_theta[0]
    id_max  = 0
    for i in range(NUM):
        if p_theta[i]> max:
            max = p_theta[i]
            id_max = i
    map_theta = id_max/NUM
    return map_theta

In [ ]:
map_theta = map_prior_x_likelihood(2, 5, 3, 10)
print(f'MAP theta: {map_theta}')

**Problem 13.2.4**. By using integration find the denominator, and hence graph the
posterior pdf.

In [ ]:
from scipy.integrate import quad

a=2
b=5
k=3
n=10

# quad() arguments: function, inferior limit, superior limit, arguments of function
# quad() returns: tuple (integral value, integral error)
pData, pData_error = quad(prior_x_likelihood, 0, 1, args=(a, b, k, n))
print(f'p(data) using integration: {pData:.3f} Error: {pData_error}') 

In [ ]:
def posterior(theta, a, b, k, n, pdata):
    prior_likelihood = prior_beta(theta, a, b) * likelihood_binom(k, n, theta)
    post = prior_likelihood / pdata
    return post

In [ ]:
def plot_posterior(a=2, b=5, k=3, n=10, pdata=1):
    fig, ax = plt.subplots(1, 1)
    theta = np.linspace(0, 1, 100)
    ax.plot(theta, posterior(theta, a, b, k, n, pdata), 'bo', ms=8, label='posterior')
    ax.set_ylabel('p(theta | data)', fontsize=11)
    ax.set_xlabel('theta', fontsize=11)

In [ ]:
plot_posterior(2, 5, 3, 10, pData)

**Problem 13.2.5.** Use your posterior to determine your break-even/fair price for participating in
the game, assuming that you win £1 if the coin comes up heads, and zero otherwise.

In [ ]:
def theta_x_posterior(theta, a, b, k, n, pdata):
    theta_x_post = theta * posterior(theta, a, b, k, n, pdata)
    return theta_x_post

In [ ]:
# This is just the mean of the posterior: INTEGRAL theta * p(theta|data) dtheta

from scipy.integrate import quad

a=2
b=5
k=3
n=10

# quad() arguments: function, inferior limit, superior limit, arguments of function
# quad() returns: tuple (integral value, integral error)
mean_ptheta, mean_ptheta_error = quad(theta_x_posterior, 0, 1, args=(a, b, k, n, pData))
print(f'posterior mean, using integration: {mean_ptheta:.3f} Error: {mean_ptheta_error}') 

# 13.3 Malarial mosquitoes

Suppose that you work for the WHO where it is your job to research the behavior of malaria-carrying mosquitoes. In particular, an important part of your research remit is to estimate adult mosquito lifespan. The lifespan of an adult mosquito is a critical determinant of the severity of
malaria, since the longer a mosquito lives the greater the chance it has of: a) becoming infected
by biting an infected human; b) surviving the period where the malarial parasite undergoes a
metamorphosis in the mosquito gut and migrates to the salivary glands; and c) passing on the
disease by biting an uninfected host.

Suppose you estimate the lifespan of mosquitoes by analyzing the results of a mark-release-recapture
field experiment. The experiment begins with the release of 1000 young adult mosquitoes (assumed to have an adult age of zero); each of which has been marked with a fluorescent die. On each day ($t$) you attempt to collect mosquitoes using a large number of traps, and count the number of marked mosquitoes that you capture ($X_t$). The mosquitoes caught each day are then re-released unharmed. The experiment goes on for 15 days in total.

Since $X_t$ is a count variable and you assume that the recapture of an individual marked mosquito
is independent and identically distributed (IID), then you choose to use a Poisson model (as an approximation to the binomial since $n$ is
large):

$X_t \sim Poisson(\lambda_t)$ <P>

$\lambda_t = 1000 \times exp(-\mu t) \psi$ <P>
    
where $\mu$ is the mortality hazard rate (assumed to be constant) and $\psi$ is the daily recapture probability. You use a $\Gamma(2, 20)$ prior for $\mu$ (which has a mean of 0.1), and a $beta(2,40)$ prior for $\psi$.

The data for the experiment is contained in the file `RWM_mosquito.csv`.

**Problem 13.3.1**. Using the data create a function that returns the likelihood.<P>
Hint: it is easiest to first write a function that accepts $(\mu, \psi)$ as an input, and outputs the mean on a day $t$.

In [ ]:
import pandas as pd

data_path = 'data/RWM_mosquito.csv'
df = pd.read_csv(data_path)

In [ ]:
df.head(16)

In [ ]:
time = df["time"].values
X    = df["recaptured"].values

print(time)
print(X)
print(type(time))

In [ ]:
# function that returns the mean number of events on day 't' -> lambda_t

def lambda_t(mu,psi,t):
    return 1000 * np.exp(-mu*t) * psi

In [ ]:
def plot_lambda_t(mu, psi, t_min, t_max):
    fig, ax = plt.subplots(1, 1)
    t = np.arange(0, 20)
    ax.plot(t, lambda_t(mu, psi, t), 'bo', ms=8, label='lambda_t')
    ax.set_ylabel('lambda_t', fontsize=11)
    ax.set_xlabel('t', fontsize=11)

In [ ]:
plot_lambda_t(0.1, 0.05, 0, 20)

In [ ]:
from scipy.stats import poisson

# function that returns the likelihood p(X_t|lambda_t)
# X_t = recaptured number of mosquitoes on day 't'

def likelihood_poisson(mu, psi, t, X):
    likelihood = 1

    for i in range(len(t)):
        lambda_i    = lambda_t(mu, psi, time[i]) 
        k_i         = X[i]
        likelihood *= poisson.pmf(k_i, lambda_i)

    return likelihood

**Problem 13.3.2**. Find the maximum likelihood estimates of $(μ, ψ)$.<P>
Hint 1: this may be easier if you create a function that returns the 
        log-likelihood, and maximise this instead.

In [ ]:
def plot_contour(param1_range, param2_range, value_grid, title_string, x_label, y_label):
    """
    Creates a 2D contour plot over two parameters.
    """

    fig, ax = plt.subplots(1,1)
    
    ax.contour(param1_range, param2_range, value_grid, levels=16, linewidths=0.5, colors='k')
    cntr = ax.contourf(param1_range, param2_range, value_grid, levels=16, cmap="RdBu_r") #cmap='magma'

    fig.colorbar(cntr, ax=ax)

    ax.set_title(title_string)

    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)

    plt.show()

In [ ]:
def mle_likelihood_poisson(t, X):
    '''
    return the MLE of mu and psi
    '''

    mu_range       = np.linspace(0.03, 0.15, 200)
    psi_range      = np.linspace(0.03, 0.1, 200)
    likelihood_mat = np.zeros((len(mu_range), len(psi_range)))

    psi_max_id     = 0
    mu_max_id      = 0
    likelihood_max = 0

    for psi in range(len(psi_range)):
        for mu in range(len(mu_range)):

            likelihood = likelihood_poisson(mu_range[mu], psi_range[psi], t, X)
            likelihood_mat[psi][mu] = likelihood
            
            if likelihood > likelihood_max:
                likelihood_max = likelihood
                psi_max_id     = psi
                mu_max_id      = mu

    plot_contour(psi_range, mu_range, likelihood_mat, 'likelihood', r"$\mu$", r"$\psi$")
    return mu_range[mu_max_id] , psi_range[psi_max_id]

In [ ]:
mle_mu, mle_psi = mle_likelihood_poisson(time, X)

print(f'MLE of mu:  {mle_mu: .4f}')
print(f'MLE of psi: {mle_psi: .4f}')

In [ ]:
def get_list_val(possible_list):

    try:
        val = possible_list[0]
    except:
        val = possible_list

    return val

**Problem 13.3.4.** Write a function for the prior, and use this to create an expression for the un-normalised posterior.

In [ ]:
from scipy.stats import beta
from scipy.stats import gamma

def psi_prior(psi):
    """
    'psi' is the probability of a mosquito dying on a certain day.
    Thus, it is appropriate to use a beta(2,40) prior.
    Since we believe that the probability of dying is small, 
    the probability is skewed towards small values and has an
    expectation of 0.047.
    """
    a = 2
    b = 40
    return get_list_val(beta.pdf(psi, a, b))

def mu_prior(mu):
    """
    'mu' is the mean mortality count on each day. 
    Thus, we use a Gamma(2,20) prior that is skewed towards small 
    average counts and has an expectation of 0.1
    """
    a = 2
    b = 1/20
    return gamma.pdf(mu, a=a, loc=0, scale=b)

In [ ]:
# The two priors are independent, so the joint prior is their product

def joint_prior(mu, psi):
    psi_p = psi_prior(psi)
    mu_p  = mu_prior(mu)
    
    prior = psi_p * mu_p
    return prior

In [ ]:
def prior_likelihood(prior, likelih):
    p_l = prior * likelih
    return p_l

**Problem 13.3.5.** Create a function that proposes a new point in parameter space using a log-normal proposal with mean at the current $\mu$ value, and a $beta(2 + \psi, 40 − \psi)$ proposal for $\psi$.<P>

Hint: use a $\text{log-N}(−0.5*\sigma^2+log(\mu)), \sigma)$, where $\mu$ is the current value of the parameter.

In [ ]:
from scipy.stats import lognorm

def log_norm_proposal(mu, sigma):
    """
    A assymetric jumpig kernel. Propose the next parameter using a
    log-Normal kernel centered on the current 'mu' parameter estimate.
    """

    mean = np.exp(np.log(mu) - 0.5 * sigma**2)
    mu_proposed = get_list_val(lognorm.rvs(s=sigma, scale=mean, size=1))

    #mean       = np.log(mu) - 0.5 * sigma**2
    #mu_proposed = get_list_val(lognorm.rvs(s=sigma, loc=mean, scale=1, size=1))

    return get_list_val(mu_proposed)


def beta_proposal(psi):
    """
    A assymetric jumpig kernel. Propose the next parameter using
    a beta kernel based on the current psi parameter estimate.
    """

    psi_proposed = np.random.beta(2 + psi, 40 - psi, 1)

    return get_list_val(psi_proposed)

**Problem 13.3.6**. Create a function that returns the ratio of the un-normalised posterior at the
proposed step location, and compares it to the current position.

In [ ]:
def proposal_current_ratio(mu, sigma, psi, t, X):
    prior       = joint_prior(mu, psi)
    likelih     = likelihood_poisson(mu, psi, t, X)
    p_l         = prior_likelihood(prior, likelih)
    
    mu_new      = log_norm_proposal(mu, sigma)
    psi_new     = beta_proposal(psi)
    
    prior_new   = joint_prior(mu_new, psi_new)
    likelih_new = likelihood_poisson(mu_new, psi_new, t, X)
    p_l_new     = prior_likelihood(prior_new, likelih_new)
    
    ratio       = p_l_new / p_l
    return mu_new, psi_new, ratio

**Problem 13.3.7**. Create a Metropolis-Hastings accept-reject function.

$r = \frac{prior\_new * likelihood\_new}{prior * likelihood} * \frac{J(mu,psi | mu\_new,psi\_new)}{J(mu\_new,psi\_new | mu,psi)}$ <P>
$ = ratio1 * \frac{J(mu,psi | mu\_new,psi\_new)}{J(mu\_new,psi\_new | mu,psi)}$ <P>
$ = ratio1 * ratio2$ <P>

In [ ]:
# compute the accpet-reject ratio

def accept_reject_ratio(mu, sigma, psi, t, X):
    
    mu_new, psi_new, ratio1 = proposal_current_ratio(mu, sigma, psi, t, X)
    denominator2 = mu_new * psi_new

    _mu_  = log_norm_proposal(mu_new, sigma)
    _psi_ = beta_proposal(psi_new)
    numerator2 = _mu_ * _psi_
    
    ratio2 = numerator2 / denominator2

    r = ratio1 * ratio2

    if (ratio1 >= 1):
        r_final = 1
    else:
        r_final = r

    return r_final

**Problem 13.3.8**. Create a Metropolis-Hastings sampler by combining your proposal and accept-reject functions.

In [ ]:
def metropolis_hastings(t, X):

    n_iterations = 4000

    psi = get_list_val(np.random.uniform(0, 1, 1))
    mu  = get_list_val(np.random.uniform(0, 1, 1))

    psi_posterior_estimate = []
    mu_posterior_estimate  = []

    sigma = 0.1

    # Used to calculate the acceptance rate
    n_accepts = 0

    burn_in = 50

    for it in range(n_iterations):

        # Proposal pdf values
        psi_new = get_list_val(beta_proposal(psi))
        mu_new  = get_list_val(log_norm_proposal(mu, sigma))

        prior_psi_new  = psi_prior(psi_new)
        prior_mu_new   = mu_prior(mu_new)
        prior_new      = prior_psi_new * prior_mu_new
        likelihood_new = likelihood_poisson(mu_new, psi_new, t, X)
        p_l_new        = prior_new * likelihood_new

        # Current pdf values
        prior_psi  = psi_prior(psi)
        prior_mu   = mu_prior(mu)
        prior      = prior_psi * prior_mu
        likelihood = likelihood_poisson(mu, psi, t, X)
        p_l        = prior * likelihood

        # The jumping probability between the current and the proposed position.
        # The Accept Reject rule
        if p_l == 0:
            print('Dividing by zero')
            
        ratio1       = p_l_new / p_l

        denominator2 = mu_new * psi_new
        _mu_         = log_norm_proposal(mu_new, sigma)
        _psi_        = beta_proposal(psi_new)
        numerator2   = _mu_ * _psi_
        ratio2       = numerator2 / denominator2

        r = ratio1 * ratio2

        if (ratio1 >= 1):
            r = 1

        a = get_list_val(np.random.uniform(0, 1, 1))

        if r > a:
            psi = psi_new
            mu  = mu_new
            n_accepts += 1

        # Store samples when the number of iterations is above the burn in
        if n_iterations > burn_in:

            psi_posterior_estimate.append(psi)
            mu_posterior_estimate.append(mu)

    print('Acceptance ratio ', n_accepts / n_iterations)

    return psi_posterior_estimate, mu_posterior_estimate


In [ ]:
def visualize_likelihood(t, X):

    psi_range = np.linspace(0, 0.1, 200)
    mu_range  = np.linspace(0, 0.15, 200)

    likelihood_matrix = np.zeros((len(mu_range), len(psi_range)))

    psi_max        = 0
    mu_max         = 0
    likelihood_max = 0

    for psi in range(len(psi_range)):
        for mu in range(len(mu_range)):
            
            likelihood_matrix[psi][mu] = likelihood_poisson(mu_range[mu], psi_range[psi], t, X)

            if likelihood_matrix[psi][mu] > likelihood_max:
                likelihood_max = likelihood_matrix[psi][mu]
                psi_max        = psi
                mu_max         = mu

    print(f'MLE psi: {psi_range[psi_max]}  MLE mu: {mu_range[mu_max]}')
    plot_contour(psi_range, mu_range, likelihood_matrix, 'likelihood', r"$\mu$", r"$\psi$")


In [ ]:
import seaborn as sns

def plot_joint_density(mu_vals, psi_vals):
    """
    plots a 2D joint density from 2 sampled ranges.
    """

    df2 = pd.DataFrame({r"$\mu$": mu_vals, r"$\psi$": psi_vals})

    sns.jointplot(data=df2, x=r"$\mu$", y=r"$\psi$", kind='kde', color="forestgreen")
    plt.show()


def plot_chains(alpha_vals, beta_vals):
    """
    Plots the evolution of the samplers accepted values.
    """
    plt.plot(alpha_vals)
    plt.plot(beta_vals)

    plt.title(r"$\mu$ (orange) :: $\psi$ (blue)")

    plt.show()

def plot_1D_density(samples, title_txt):
    """
    plots a 1D density from a sampled range.
    """

    #sns.distplot(samples, hist=True, kde=True, bins=int(20), color="orange")
    sns.histplot(data=samples, stat='probability', discrete=False, kde=True, bins=int(20), color="orange")
    plt.title(title_txt)
    plt.show()


In [ ]:
# Visualize the likelihood P(data|mu,psi)

visualize_likelihood(time, X)

# Run the Metropolis-Hastings algorithm

psi_posterior_estimate, mu_posterior_estimate = metropolis_hastings(time, X)

# Plot the joint probability P(mu,psi)

plot_joint_density(mu_posterior_estimate, psi_posterior_estimate)

# Plot the parameters (mu,psi) chains/paths followed by the Metropolis-Hastings algorithm

plot_chains(psi_posterior_estimate, mu_posterior_estimate)

# Plot the posterior probability P(mu | data)

plot_1D_density(mu_posterior_estimate, r"$\mu$")

# Plot the posterior probability P(psi | data)

plot_1D_density(psi_posterior_estimate, r"$\psi$")